# FoundML 3 — Coding Assignment (Student Version)
## Transfer Learning with a Pretrained CNN — TensorFlow/Keras (CIFAR-10)

You will practice **transfer learning** using a pretrained CNN as a feature extractor, then **fine-tune** it.
The dataset is **CIFAR-10** (10 classes).

Difficulty level: **moderate**.
- You will complete several short Keras code blocks (model construction, compile/train/evaluate).
- You will not implement any deep learning primitives from scratch.

All numerical answers must be rounded to **2 decimals**.


## Rules
- Use the provided random seeds and hyperparameters.
- Do not change the grading variable names (Q1–Q4).
- Keep training light: the specified epochs are small by design.


## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(0)
tf.random.set_seed(0)


## 1. Load CIFAR-10 and visualize samples

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.squeeze()
y_test = y_test.squeeze()

X_train.shape, y_train.shape, X_test.shape, y_test.shape


In [ ]:
class_names = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]

def show_samples(X, y, n=10, title="Samples"):
    fig, axes = plt.subplots(1, n, figsize=(1.5*n, 2))
    fig.suptitle(title)
    for i, ax in enumerate(axes):
        ax.imshow(X[i])
        ax.set_title(class_names[int(y[i])], fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_samples(X_train, y_train, n=10, title="CIFAR-10: first 10 training images")


## 2. Preprocess for a pretrained network

We will use **MobileNetV2** pretrained on ImageNet.
Requirements:
- Resize images to `96x96`
- Convert labels to one-hot
- Use the correct preprocessing function for MobileNetV2


In [ ]:
IMG_SIZE = 96
NUM_CLASSES = 10

# Write here your own code.
# Requirements:
# - Create tf.data datasets: train_ds, test_ds
# - Resize to (IMG_SIZE, IMG_SIZE)
# - Convert to float and apply: tf.keras.applications.mobilenet_v2.preprocess_input
# - One-hot encode labels to NUM_CLASSES
# - Use batch_size = 64
# - Shuffle only training set (buffer size >= 1000)
#
# YOUR CODE HERE


# SANITY CHECK (do not modify)

In [ ]:
# SANITY CHECK (do not modify)
assert 'train_ds' in globals() and 'test_ds' in globals(), "train_ds/test_ds not defined"
for xb, yb in train_ds.take(1):
    assert xb.shape[1:3] == (IMG_SIZE, IMG_SIZE), "Images must be resized to IMG_SIZE"
    assert yb.shape[-1] == NUM_CLASSES, "Labels must be one-hot with NUM_CLASSES"
print("Sanity check passed: datasets look valid.")


## 3. Build a transfer learning model (feature extractor)

Use:
- `base_model = MobileNetV2(include_top=False, weights="imagenet", input_shape=(96,96,3))`
- Freeze the base model (`base_model.trainable = False`)
- Add a small classification head:
  - GlobalAveragePooling2D
  - Dense(128, relu)
  - Dense(NUM_CLASSES, softmax)

Compile with:
- Adam(learning_rate=1e-3)
- loss = categorical_crossentropy
- metric = accuracy


In [ ]:
# Write here your own code.
# Requirements:
# - define: base_model, model_frozen
# - compile: model_frozen
#
# YOUR CODE HERE


# SANITY CHECK (do not modify)

In [ ]:
# SANITY CHECK (do not modify)
assert 'model_frozen' in globals(), "model_frozen not defined"
assert model_frozen.trainable, "Top model should be trainable"
# Base model must be frozen
base = None
for layer in model_frozen.layers:
    if isinstance(layer, keras.Model):
        base = layer
        break
# Not all Keras graphs expose it as nested model; so we check any MobileNetV2 layers are non-trainable
trainable_in_base = any(("mobilenetv2" in l.name.lower()) and l.trainable for l in model_frozen.layers)
print("Sanity check: trainable_in_base_layers =", trainable_in_base)


## 4. Train the frozen model (feature extractor stage)

Train for:
- epochs = 3

Store the Keras History in `history_frozen`.


In [ ]:
# Write here your own code.
# Requirements:
# - fit model_frozen on train_ds, validate on test_ds
# - epochs = 3
# - store History in: history_frozen
#
# YOUR CODE HERE


### **Question 1**
What is the **test accuracy** after training the frozen model for 3 epochs?

- Use `model_frozen.evaluate(test_ds, verbose=0)`
- Enter a single float rounded to **2 decimals**.


In [ ]:
# ANSWER Q1
# Define: Q1_test_acc_frozen
#
# YOUR CODE HERE


## 5. Fine-tune: unfreeze the top layers of the base model

Fine-tuning setup:
- Unfreeze the base model
- Keep BatchNorm layers frozen (common practice for stability)
- Fine-tune only the **last 20 layers** of the base model (except BatchNorm)
- Compile with a lower learning rate: Adam(learning_rate=1e-5)
- Train for epochs = 2

Store History in `history_finetune`.


In [ ]:
# Write here your own code.
# Requirements:
# - unfreeze base model layers appropriately
# - recompile model
# - train for 2 epochs
# - store History in: history_finetune
#
# YOUR CODE HERE
# IMPORTANT: store the returned History object in a variable named `history_finetune`


### **Question 2**
What is the **test accuracy** after fine-tuning for 2 epochs?

- Use `model_frozen.evaluate(test_ds, verbose=0)` (same model object)
- Enter a single float rounded to **2 decimals**.


In [ ]:
# ANSWER Q2
# Define: Q2_test_acc_finetuned
#
# YOUR CODE HERE


## 6. Model size (trainable parameters)

Report the number of **trainable parameters** after fine-tuning is configured (i.e., after unfreezing last layers).

Hint:
- `model_frozen.count_params()` gives total params
- You can compute trainable params by summing `np.prod(v.shape)` for `model_frozen.trainable_weights`


In [ ]:
# Write here your own code.
# Requirements:
# - compute trainable_params (int)
#
# YOUR CODE HERE


### **Question 3**
How many **trainable parameters** does the model have after fine-tuning is configured?

- Enter an integer.


In [ ]:
# ANSWER Q3
# Define: Q3_trainable_params_after_finetune
#
# YOUR CODE HERE


### **Question 4**
How many **trainable parameters** does the model have in the **frozen feature-extractor stage** (before fine-tuning)?

- Compute this right after building `model_frozen` (before unfreezing any base layers).
- Enter an integer.


In [ ]:
# ANSWER Q4
# Requirements:
# - compute trainable_params_frozen_stage (int) from model_frozen.trainable_weights
# - define: Q4_trainable_params_frozen_stage
#
# YOUR CODE HERE


## 7. Visualization (not graded)

Plot training curves for accuracy (frozen stage + fine-tune stage). A simple plot is enough.


In [ ]:
import matplotlib.pyplot as plt

# Write here your own code.
# Requirements:
# - plot training accuracy from history_frozen and history_finetune
#
# YOUR CODE HERE
# IMPORTANT: store the returned History object in a variable named `history_finetune`


## Submission checklist

Your notebook must define:
- `Q1_test_acc_frozen`
- `Q2_test_acc_finetuned`
- `Q3_trainable_params_after_finetune`
- `Q4_trainable_params_frozen_stage`
